# Homework №2

*Natural Language Processing Course, HSE 2026*

### General Rules and Submission Guidelines

1. Copying code from external sources (**including using LLMs**) without explicit citation is strictly prohibited and will result in 0 points for the entire assignment. If you consult any resources or AI tools, you must clearly state this in a separate Markdown cell. If suspected of this, you might be asked to explain your code to the grader and answer their questions during a separate session to avoid the mentioned penalty.
2. All results must be fully **reproducible**. You are required to use `set_seed` everywhere so that the grader can obtain the same results when rerunning your notebook.
3. The notebook must run from top to bottom without errors. Submissions that fail to execute sequentially will not be accepted.
4. You must satisfy all requirements in each task to receive full credit. Partial completion may lead to partial scoring.
5. Do not modify the original notebook structure or provided Markdown cells. You are only allowed to write code in the sections marked `# TODO: your code here`. Any explanations, interpretations, or additional comments must be placed **in separate Markdown cells**. If you choose to do so, leave an explanation as to what and why was changed.
6. The final submission must be a completed `.ipynb` Jupyter Notebook. You may conduct your experiments in Jupyter Notebook, VS Code, or Google Colab — whichever environment you prefer.

**Note: You may write your textual comments in Markdown for the conducted experiments *in either Russian or English***.

### Environment Setup

Loading necessary libraries. If you need anything else, feel free to add more libraries and dependencies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from datasets import Dataset
import random

from transformers import AutoModelForCausalLM, set_seed

seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### **0. Introduction to the Task and Data**

In this assignment, you will get acquainted with a binary text classification task in Russian, known as Linguistic Acceptability. Your goal will be to train a model capable of distinguishing grammatically correct sentences from incorrect ones.

**RuCoLA (Russian Corpus of Linguistic Acceptability)** is a corpus of Russian sentences annotated with a binary label indicating whether a sentence is linguistically acceptable (grammatically correct and meaningful) or not.

**Structure:** The data is already split into training, validation, and test sets. A detailed description and the dataset itself are available in the project's official [repository](https://github.com/RussianNLP/RuCoLA).

***Important:*** Before starting the assignment, carefully study the repository's `README`. Pay special attention to the sections describing the data structure and the already implemented baselines — this will help you in your work.

### **1. Data Preparation (0.25 point)**



**Data Location:**
The data is located in the [repository](https://github.com/RussianNLP/RuCoLA/tree/main/data). Download the files contained in this folder.




**Files to be Used:**
*   For **training** (`train`), we will use the file **`in_domain_train.csv`**. This file contains examples from linguistic sources.
*   For **testing** (`test`), we will use the file **`in_domain_dev.csv`**.






**Data Format:**
The essential information for us is located in two columns:
*   **`sentence`**: This column contains the text of the sentence (input data for the model).
*   **`acceptable`**: This is the target variable for prediction. It contains a binary label:
    *   **1** — the sentence is linguistically acceptable (grammatically correct).
    *   **0** — the sentence is unacceptable (contains an error).

By the end of this step, you should **have both training and test datasets loaded and ready for further preprocessing** and model experimentation.

In [ ]:
train_url = "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_train.csv"
dev_url = "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_dev.csv"

train_df = pd.read_csv(train_url)
dev_df = pd.read_csv(dev_url)

train_df = train_df[["sentence", "acceptable"]]
dev_df = dev_df[["sentence", "acceptable"]]

print(train_df.shape, dev_df.shape)
train_df.head()

**Evaluation Metrics:**
The performance of your models will be evaluated using:

1.  **Accuracy:** The percentage of sentences for which the model correctly predicted the label.
2.  **Matthews Correlation Coefficient (MCC):** This is a more robust metric for binary classification tasks, especially if the classes might be slightly imbalanced. It takes into account all four categories of the confusion matrix (True Positives, True Negatives, False Positives, False Negatives) and returns a value in the range of -1 to +1.

**Important:** You *do not need to implement MCC* from scratch. Import and use the ready-made function from the scikit-learn library:

In [ ]:
from sklearn.metrics import matthews_corrcoef

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    mcc = matthews_corrcoef(labels, preds)
    return {"accuracy": acc, "mcc": mcc}

### **2. Baseline: Fine-tuning ruBERT (1 point)**


In this section, you will implement a classifier based on one of the BERT family models adapted for the Russian language. Perform fine-tuning of the model on the training data `in_domain_train.csv` and evaluate its quality on `in_domain_dev.csv`.

**What you need to do:**

1.  **Model Selection (0.25 point)**

For the experiment, you can choose one of the following pre-trained models from the Hugging Face Hub (they differ in size and speed):
*   [`cointegrated/rubert-tiny2`](https://huggingface.co/cointegrated/rubert-tiny2)
*   [`ai-forever/ruBert-base`](https://huggingface.co/ai-forever/ruBert-base)
*   [`ai-forever/ruBert-large`](https://huggingface.co/ai-forever/ruBert-large)



In [ ]:
model_name = "cointegrated/rubert-tiny2"

bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

2.  **Data Preparation (0.25 point)**
    
    Convert the texts and labels into a format suitable for `transformers`. Use the tokenizer corresponding to your chosen model.

In [ ]:
train_ds = Dataset.from_pandas(train_df)
dev_ds = Dataset.from_pandas(dev_df)

train_ds = train_ds.rename_column("sentence", "text").rename_column("acceptable", "labels")
dev_ds = dev_ds.rename_column("sentence", "text").rename_column("acceptable", "labels")

if "__index_level_0__" in train_ds.column_names:
    train_ds = train_ds.remove_columns("__index_level_0__")
if "__index_level_0__" in dev_ds.column_names:
    dev_ds = dev_ds.remove_columns("__index_level_0__")

def tok_fn(batch):
    return bert_tokenizer(batch["text"], truncation=True)

train_ds = train_ds.map(tok_fn, batched=True)
dev_ds = dev_ds.map(tok_fn, batched=True)

train_ds = train_ds.remove_columns(["text"])
dev_ds = dev_ds.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

3. **Fine-tuning (0.25 point)**

Stopping Criterion: Train the model until it reaches a plateau (early stopping by loss). Once the loss stops improving for several epochs (e.g., 2-3), training should be stopped. This helps prevent overfitting.
Record the number of epochs it took to reach the plateau.

In [ ]:
training_args = TrainingArguments(
    output_dir="rubert_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=seed,
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
print("stopped at epoch:", trainer.state.epoch)

4. **Quality Evaluation (0.25 point)**

After training is complete, make predictions for the entire in_domain_dev.csv file.
Calculate the Accuracy and MCC metrics (using sklearn.metrics.matthews_corrcoef).
Compare the obtained results and comment on them in a Markdown cell.

In [ ]:
pred = trainer.predict(dev_ds)
logits = pred.predictions
y_true = pred.label_ids

y_pred = np.argmax(logits, axis=1)

acc = (y_pred == y_true).mean()
mcc = matthews_corrcoef(y_true, y_pred)

print("Accuracy:", acc)
print("MCC:", mcc)

**Your answer here:** ...

### **3. Zero-shot Approach Using a Generative Model (ruGPT-3) (4.25 point)**


In this section, we will try to solve the task without any training, using only a pre-trained generative language model — **ruGPT-3 Large**.



**Model:** Use [`ai-forever/rugpt3large_based_on_gpt2`](https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2) from the `transformers` library.



**Task:**

1.  **Implement a function to calculate loss for a single text (1.25 point)**

You will need to write a function that:
*   Tokenizes the input text, preparing it for the model (don't forget `return_tensors="pt"` and moving it to the `device`).
*   Passes the tokenized inputs to the model, specifying `labels` (for GPT-2/3 models, labels are usually the same as `input_ids`, since the task is next-token prediction).
*   Extracts the `loss` value from the model's output.
*   Returns the loss value (a number).

    **Important:** To complete this task, you can use [this Colab notebook](https://colab.research.google.com/drive/1BnfhKQQiNAXrKlkIynVPFleQ-epF3I55#scrollTo=fOemj3PbeCZi) and the `transformers` documentation as a reference.



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")
model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")

In [ ]:
def get_loss_num(text):
    '''
    Tokenize the input text and move it to the specified device
    Shift the inputs to create labels for the next-token prediction task
    Move labels to the correct device if you are using GPU
    Calculate loss
    '''
    model.to(device)
    model.eval()

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs, labels=inputs["input_ids"])

    return float(out.loss)

2.  **Implement the prediction function `predict_zero_shot(text, pos_template, neg_template)` (1 point)**
    *   The function takes the original text and two templates (prompts) — one for the positive (acceptable) class and one for the negative (unacceptable) class.
    *   The templates should be strings with a placeholder `{}`, where the original text will be inserted. For example: `"This sentence is grammatically correct: {}"`.
    *   Inside the function, calculate the loss for the text inserted into the positive template (`pos_loss`) and the loss for the text inserted into the negative template (`neg_loss`).
    *   Compare the values: if `pos_loss` is **less than** `neg_loss` (the model finds the positive variant more probable), return `1` (the "acceptable" class); otherwise, return `0`.

In [ ]:
def predict_zero_shot(text, pos_template, neg_template):
    pos_text = pos_template.format(text)
    neg_text = neg_template.format(text)

    pos_loss = get_loss_num(pos_text)
    neg_loss = get_loss_num(neg_text)

    if pos_loss < neg_loss:
        return 1
    else:
        return 0

3.  **Experiments with prompts and generation parameters (1 point)**

This part consists of two sub-tasks: first, you will find the best prompt pair using a fixed decoding strategy. Then, you will experiment with different generation parameters using that best prompt.

Fix the decoding strategy to greedy search. Use this setting for all prompt evaluations.
Create at least three different pairs of prompts. Vary the wording, punctuation, level of detail, and whether the prompt is a statement, question, or instruction.

For each prompt compute Accuracy and MCC.

Identify the prompt pair that yields the highest performance. This will be your best prompt for the next sub-task.

In [ ]:
prompt_pairs = [
    ("pair1", "Это грамматически правильное предложение: {}", "Это грамматически неправильное предложение: {}"),
    ("pair2", "Предложение корректно? {}.", "Предложение некорректно? {}."),
    ("pair3", "Верное предложение: {}", "Неверное предложение: {}"),
]

y_true = dev_df["acceptable"].tolist()
sentences = dev_df["sentence"].tolist()

results = []

for name, pos_t, neg_t in prompt_pairs:
    preds = []
    for i, s in enumerate(sentences):
        preds.append(predict_zero_shot(s, pos_t, neg_t))
        if (i + 1) % 200 == 0:
            print(name, "done", i + 1)

    acc = (np.array(preds) == np.array(y_true)).mean()
    mcc = matthews_corrcoef(y_true, preds)
    results.append({"name": name, "accuracy": acc, "mcc": mcc})

results_df = pd.DataFrame(results).sort_values("mcc", ascending=False)
print(results_df)

best_pair = results_df.iloc[0]["name"]
best_pos = [p[1] for p in prompt_pairs if p[0] == best_pair][0]
best_neg = [p[2] for p in prompt_pairs if p[0] == best_pair][0]

print("best:", best_pair)

Using the best prompt from previous step, experiment with at least three different generation strategies. The loss calculation can be affected by how the model processes the input. You can modify the generation parameters even though you are only computing loss

For each configuration, evaluate on the test set and record Accuracy and MCC

In [ ]:
def get_loss_num_cfg(text, max_len=None, trunc_side="right"):
    old_side = tokenizer.truncation_side
    tokenizer.truncation_side = trunc_side

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )
    tokenizer.truncation_side = old_side

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])

    return float(out.loss)

def predict_zero_shot_cfg(text, pos_t, neg_t, max_len=None, trunc_side="right"):
    pos_loss = get_loss_num_cfg(pos_t.format(text), max_len=max_len, trunc_side=trunc_side)
    neg_loss = get_loss_num_cfg(neg_t.format(text), max_len=max_len, trunc_side=trunc_side)
    return 1 if pos_loss < neg_loss else 0

configs = [
    ("full", None, "right"),
    ("max128_right", 128, "right"),
    ("max128_left", 128, "left"),
]

y_true = dev_df["acceptable"].tolist()
sentences = dev_df["sentence"].tolist()

gen_results = []

for name, max_len, side in configs:
    preds = []
    for i, s in enumerate(sentences):
        preds.append(predict_zero_shot_cfg(s, best_pos, best_neg, max_len=max_len, trunc_side=side))
        if (i + 1) % 200 == 0:
            print(name, "done", i + 1)

    acc = (np.array(preds) == np.array(y_true)).mean()
    mcc = matthews_corrcoef(y_true, preds)
    gen_results.append({"cfg": name, "accuracy": acc, "mcc": mcc})

gen_df = pd.DataFrame(gen_results).sort_values("mcc", ascending=False)
print(gen_df)

4.  **Analysis of Results (1 point)**
    *   Compare the metrics obtained with different prompts.
    *   Draw conclusions about how the following factors influence quality:
        *   The wording of the task (statement, question, instruction).
        *   The presence of punctuation (a period at the end, a question mark).
        *   The length and specificity of the prompt.
    *   Try to explain why one pair of prompts worked better than another.

**Your answer here:** ...

### **4. Instruction-Tuned Model: Qwen2.5-Instruct (2 points)**



In this section, you will work with the **Qwen2.5-Instruct** model. Your goal is to select prompts that will allow the model to solve the binary classification task of linguistic acceptability without any fine-tuning, relying solely on its ability to follow instructions.



**Model:**
*   Use the base instruction-tuned version: [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).
*   *Note:* You may also experiment with larger versions (e.g., `Qwen/Qwen2.5-7B-Instruct`) and compare the results.


**Task:**

1.  **Design a prompt for the task (0.5 point)**

Create at least three different prompts that set various contexts, roles, or behavioral constraints for the model and ask the model to classify a sentence as acceptable (1) or unacceptable (0). The prompt should be passed to the model together with the input sentence.

**!Do not use a system prompt!** in this part - only a single user message or a direct instruction.


In [ ]:
# free ruGPT-3 to save memory (we don't need it дальше)
try:
    del model
    del tokenizer
except:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()

qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_name, torch_dtype=dtype)
qwen_model.to(device)
qwen_model.eval()

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_prompts = [
    ("ru_short",
     "Определи, грамматически корректно ли предложение. 1 - да, 0 - нет.\nПредложение: {}\nОтвет:"),
    ("en_short",
     "Classify the sentence as acceptable (1) or unacceptable (0). Answer with one digit.\nSentence: {}\nAnswer:"),
    ("ru_example",
     "Ты лингвист. Пример: «Я читаю книгу.» -> 1. Теперь оцени: {}. Ответ только 0 или 1:")
]

qwen_gen_kwargs = {"max_new_tokens": 20, "do_sample": False}

def qwen_generate(sentence, prompt_template, system_prompt=None):
    if system_prompt is None:
        messages = [{"role": "user", "content": prompt_template.format(sentence)}]
    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt_template.format(sentence)}
        ]

    chat = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(chat, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out_ids = qwen_model.generate(**inputs, **qwen_gen_kwargs)
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(gen_ids, skip_special_tokens=True)

2.  **Implement response post-processing (0.5 point)**

The model generates free text, so you need to extract the predicted label (0 or 1) from its output. Write a function that:
    *   Takes the generated model text as input.
    *   Extracts the integer 0 or 1 from it. Handle cases where the model provides additional explanations (e.g., look for the first occurrence of "1" or "0", use simple regular expressions).
    *   If the response is ambiguous, decide on a processing strategy. Describe your approach.


In [ ]:
import re

def extract_label(text):
    m = re.search(r"\b([01])\b", text)
    if m:
        return int(m.group(1))

    m = re.search(r"[01]", text)
    if m:
        return int(m.group())

    return 0

3.  **Experiments with prompts (0.5 point)**

Conduct experiments by varying prompt characteristics. For each variant, evaluate the model on the entire `in_domain_dev.csv` file and compute **Accuracy** and **MCC**. Consider at least the following dimensions:



  *   **Prompt Language:** prompts in Russian vs. prompts in English. How does language proficiency affect the result?
  *   **Instruction Style:** compare different formulations.
  *   **Prompt Length and Detail:** short vs. detailed instructions; you can try adding one example (few-shot).




In [ ]:
y_true = dev_df["acceptable"].tolist()
sentences = dev_df["sentence"].tolist()

qwen_results = []

for name, prompt_t in qwen_prompts:
    preds = []
    for i, s in enumerate(sentences):
        gen_text = qwen_generate(s, prompt_t)
        preds.append(extract_label(gen_text))
        if (i + 1) % 50 == 0:
            print(name, "done", i + 1)

    acc = (np.array(preds) == np.array(y_true)).mean()
    mcc = matthews_corrcoef(y_true, preds)
    qwen_results.append({"name": name, "accuracy": acc, "mcc": mcc})

qwen_df = pd.DataFrame(qwen_results).sort_values("mcc", ascending=False)
print(qwen_df)

best_qwen_name = qwen_df.iloc[0]["name"]
best_qwen_prompt = [p[1] for p in qwen_prompts if p[0] == best_qwen_name][0]

print("best prompt:", best_qwen_name)


4.  **Analysis of Results (0.5 point)**
    *   Compare the quality of different prompts. Which ones work better and why?
    *   Draw conclusions about the influence of the instruction language.
    *   Provide your thoughts on whether the model size is sufficient to solve this task with good quality.
    *   Describe your findings in MarkDown.

**Your answer here:** ...

### **5. Adding a System Prompt (0,75 point)**


Add a System Prompt and **conduct an experiment similar to** **part 4**, varying the System Prompt. Draw conclusions about the influence of the System Prompt. Create **at least three different system prompts** that set various contexts, roles, or behavioral constraints for the model.

In [ ]:
system_prompts = [
    "Ты строгий лингвист. Отвечай только цифрой 0 или 1, без объяснений.",
    "You are a classifier. Output only 0 or 1. No extra text.",
    "Отвечай максимально коротко: только 0 или 1. Не добавляй слова."
]

y_true = dev_df["acceptable"].tolist()
sentences = dev_df["sentence"].tolist()

sys_results = []

for sys_p in system_prompts:
    preds = []
    for i, s in enumerate(sentences):
        gen_text = qwen_generate(s, best_qwen_prompt, system_prompt=sys_p)
        preds.append(extract_label(gen_text))
        if (i + 1) % 50 == 0:
            print("sys done", i + 1)

    acc = (np.array(preds) == np.array(y_true)).mean()
    mcc = matthews_corrcoef(y_true, preds)
    sys_results.append({"system_prompt": sys_p, "accuracy": acc, "mcc": mcc})

sys_df = pd.DataFrame(sys_results).sort_values("mcc", ascending=False)
print(sys_df[["accuracy", "mcc"]])

best_system_prompt = sys_df.iloc[0]["system_prompt"]
print("best system prompt:", best_system_prompt)

**Your answer here:** ...

### **6. Base (Non-Instruction) Version of Qwen2.5 (0,75 point)**


Conduct an experiment similar to the approach in section 4-5, but using the base non-instruction version of the **Qwen/Qwen2.5-1.5B** model. Use [`Qwen/Qwen2.5-1.5B`](https://huggingface.co/Qwen/Qwen2.5-1.5B) (or `Qwen/Qwen2.5-7B`).

Use exactly the same hyperparameters, generation strategy, and system prompts that turned out to be optimal for the instruction-tuned version.

In [ ]:
# free instruct model to save memory
try:
    del qwen_model
except:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()

qwen_base_name = "Qwen/Qwen2.5-1.5B"

qwen_base_tokenizer = AutoTokenizer.from_pretrained(qwen_base_name)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
qwen_base_model = AutoModelForCausalLM.from_pretrained(qwen_base_name, torch_dtype=dtype)
qwen_base_model.to(device)
qwen_base_model.eval()

if qwen_base_tokenizer.pad_token is None:
    qwen_base_tokenizer.pad_token = qwen_base_tokenizer.eos_token

def qwen_base_generate(sentence):
    if getattr(qwen_base_tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": best_system_prompt},
            {"role": "user", "content": best_qwen_prompt.format(sentence)}
        ]
        text = qwen_base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = best_system_prompt + "\n" + best_qwen_prompt.format(sentence)

    inputs = qwen_base_tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out_ids = qwen_base_model.generate(**inputs, **qwen_gen_kwargs)
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    return qwen_base_tokenizer.decode(gen_ids, skip_special_tokens=True)

y_true = dev_df["acceptable"].tolist()
sentences = dev_df["sentence"].tolist()

preds = []
for i, s in enumerate(sentences):
    gen_text = qwen_base_generate(s)
    preds.append(extract_label(gen_text))
    if (i + 1) % 50 == 0:
        print("base done", i + 1)

acc = (np.array(preds) == np.array(y_true)).mean()
mcc = matthews_corrcoef(y_true, preds)

print("Qwen base Accuracy:", acc)
print("Qwen base MCC:", mcc)

**Your answer here:** ...

### **7. Draw conclusions (1 point)**


Describe whether the results of the instruction-tuned and non-instruction versions of the same base model differ. Explain what these differences (or lack thereof) are related to. How does instruction fine-tuning affect the model's ability to solve the linguistic acceptability task?

**Your answer here:** ...